In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

sys.path.append(os.path.dirname(os.getcwd()))
from lib.utils import get_sequence_data
from lib.BLogistic import SkewedBLogistic

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

folder_path = r"../../MarketData/historical_data"
context_window = 60
X, Y     = get_sequence_data(folder_path, context_window, force_recompute=True)
dof      = 16
simple_X = torch.tensor(X[:, :, 0], device=device)
simple_Y = torch.tensor(Y, device=device).reshape(-1, 1)

dev_size = 10000
np.random.seed(0)
indices = np.random.permutation(simple_X.shape[0])

dev_indices = indices[:dev_size]
train_indices = indices[dev_size:]
train_X = simple_X[train_indices]
train_Y = simple_Y[train_indices]
dev_X = simple_X[dev_indices, :]
dev_Y = simple_Y[dev_indices]

std = train_Y.std()
train_X = train_X / std
train_Y = train_Y / std
dev_X = dev_X / std
dev_Y = dev_Y / std
print("train_X", train_X.shape, "train_Y", train_Y.shape, "dev_X", dev_X.shape, "dev_Y", dev_Y.shape)
print("std", std, train_X.std())

skipping 2020-11-27
skipping 2020-12-24
skipping 2021-11-26
skipping 2022-11-25
skipping 2023-07-03
skipping 2023-11-24
skipping 2024-07-03
skipping 2024-11-29
skipping 2024-12-24
skipping 2025-07-03
train_X torch.Size([402426, 60]) train_Y torch.Size([402426, 1]) dev_X torch.Size([10000, 60]) dev_Y torch.Size([10000, 1])
std tensor(0.0004, device='cuda:0') tensor(1.0232, device='cuda:0')


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class LSTMProbNN(nn.Module):
    def __init__(self, input_size, device):
        super(LSTMProbNN, self).__init__()
        self.input_size = input_size
        self.device = device

        print(f"\nInitializing LSTM with input_size={input_size}")

        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=128,
            batch_first=True
        )
        self.dropout1 = nn.Dropout(0.02)

        self.lstm2 = nn.LSTM(
            input_size=128,
            hidden_size=64,
            batch_first=True
        )
        self.dropout2 = nn.Dropout(0.02)

        self.lstm3 = nn.LSTM(
            input_size=64,
            hidden_size=32,
            batch_first=True
        )
        self.dropout3 = nn.Dropout(0.02)

        self.fc = nn.Linear(32, dof)
        nn.init.uniform_(self.fc.weight, -0.01, 0.01)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)  # Add feature dimension

        x, _ = self.lstm1(x)
        x = self.dropout1(x)

        x, _ = self.lstm2(x)
        x = self.dropout2(x)

        x, _ = self.lstm3(x)
        x = self.dropout3(x)

        x = x[:, -1, :]

        params = self.fc(x)
        return params

    def loss_fn(self, x, y):
        """
        Skewed Student's t Negative Log-Likelihood Loss
        Based on Equation 16 from the paper
        """
        params = self.forward(x)

        # Extract parameters
        mu = params[:, 0]
        log_sigma = params[:, 1]
        sigma = torch.exp(log_sigma)  # Ensure positive
        nu = 2.0 + torch.nn.functional.softplus(params[:, 2])  # Ensure ν > 2
        log_xi = params[:, 3]
        xi = torch.exp(log_xi)  # Ensure ξ > 0

        # Standardize
        y = y.squeeze()
        z = (y - mu) / sigma

        # Compute skewed Student's t log-likelihood (Equation 16)
        # Part 1: log(2/(ξ + 1/ξ))
        term1 = torch.log(torch.tensor(2.0, device=self.device)) - torch.log(xi + 1.0/xi)

        # Part 2: -log(σ)
        term2 = -log_sigma

        # Part 3: Piecewise Student's t with Heaviside function
        # For z < 0: use fSt(ξ*z; 0, 1, ν)
        # For z >= 0: use fSt(z/ξ; 0, 1, ν)

        # Student's t log PDF: log fSt(z; 0, 1, ν)
        def student_t_logpdf(z, nu):
            log_gamma_term = torch.lgamma((nu + 1) / 2) - torch.lgamma(nu / 2)
            log_constant = -0.5 * torch.log(nu * np.pi)
            log_density = -(nu + 1) / 2 * torch.log(1 + z**2 / nu)
            return log_gamma_term + log_constant + log_density

        # Apply skewness transformation
        z_left = xi * z  # For z < 0
        z_right = z / xi  # For z >= 0

        logpdf_left = student_t_logpdf(z_left, nu)
        logpdf_right = student_t_logpdf(z_right, nu)

        # Heaviside function: select appropriate PDF
        mask_negative = (z < 0).float()
        mask_positive = (z >= 0).float()

        # Combine using Heaviside
        log_pdf_z = torch.log(
            torch.exp(logpdf_left) * mask_negative +
            torch.exp(logpdf_right) * mask_positive + 1e-10
        )

        # Total log-likelihood
        log_likelihood = term1 + term2 + log_pdf_z

        return -log_likelihood.mean()


In [3]:

def train_lstm_nn(
    train_X, train_Y, dev_X, dev_Y,
    lr, num_epochs, device,
    input_size=1,
    batch_size=128
):
    print(f"Train X shape: {train_X.shape}, Train Y shape: {train_Y.shape}")
    print(f"Using batch_size={batch_size}, lr={lr}, input_size={input_size}")

    # Initialize model
    model = LSTMProbNN(input_size, device).to(device)

    # Optimizer with L2 regularization
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.002)

    # Training tracking
    train_losses, dev_losses = [], []
    best_dev_loss = float('inf')
    best_model_state = None

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        n_batches = 0

        # Shuffle training data
        indices = torch.randperm(train_X.shape[0])
        train_X_shuffled = train_X[indices]
        train_Y_shuffled = train_Y[indices]

        for i in range(0, train_X.shape[0], batch_size):
            batch_X = train_X_shuffled[i:i+batch_size]
            batch_Y = train_Y_shuffled[i:i+batch_size]

            if batch_X.shape[0] == 0:
                continue

            # Forward pass
            loss = model.loss_fn(batch_X, batch_Y)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            n_batches += 1

        avg_train_loss = running_loss / max(n_batches, 1)

        # Validation every 5 epochs
        if epoch % 5 == 0 or epoch == num_epochs - 1:
            model.eval()
            with torch.no_grad():
                dev_loss_sum = 0.0
                n_dev_batches = 0

                for j in range(0, dev_X.shape[0], batch_size):
                    batch_X = dev_X[j:j+batch_size]
                    batch_Y = dev_Y[j:j+batch_size]

                    if batch_X.shape[0] == 0:
                        continue

                    dev_loss_sum += model.loss_fn(batch_X, batch_Y).item()
                    n_dev_batches += 1

                avg_dev_loss = dev_loss_sum / max(n_dev_batches, 1)

            train_losses.append(avg_train_loss)
            dev_losses.append(avg_dev_loss)


    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with Dev Loss: {best_dev_loss:.4f}")

    return model, train_losses, dev_losses


In [4]:
print(f"Context window: {context_window}")
print(f"DOF: {dof}")
print(f"Device: {device}")

# Training
lr = 2e-4
num_steps = 3000
model, train_losses, dev_losses = train_lstm_nn(train_X, train_Y, dev_X, dev_Y, lr, num_steps, device=device)
torch.save(model.state_dict(), f"jl_lstm_model_context{context_window}.pth")

# Loading and plotting
model = LSTMProbNN(context_window, dof, device).to(device)
model.load_state_dict(torch.load(f"jl_ltsm_model_context{context_window}.pth", map_location=device))
model.eval()

Context window: 60
DOF: 16
Device: cuda
Train X shape: torch.Size([402426, 60]), Train Y shape: torch.Size([402426, 1])
Using batch_size=128, lr=0.0002, input_size=1

Initializing LSTM with input_size=1


KeyboardInterrupt: 

In [13]:

plot_xs = torch.linspace(-8, 8, 10000, dtype=torch.float32, device='cpu')
nplots = min(10, dev_X.shape[0])

plt.figure(figsize=(10, 6))
with torch.no_grad():
    for idx in range(nplots):
        color = plt.cm.viridis(idx / nplots)

        # Ensure float32 and move to CPU for plotting
        dev_X_cpu = dev_X[idx, :].reshape(1, -1).float().cpu()
        dev_Y_cpu = dev_Y[idx, :].float().cpu()

        # Use batched computation
        plot_ys = model.get_pdf_batched(dev_X_cpu, plot_xs, batch_size=500)
        plt.plot(plot_xs.numpy(), plot_ys.numpy(), color=color, alpha=0.7)

        point_pdf = model.get_pdf(dev_X_cpu, dev_Y_cpu.reshape(-1))
        plt.scatter(dev_Y_cpu.numpy(), point_pdf.cpu().numpy(), color=color, s=50, zorder=5)

plt.xlabel("Return")
plt.ylabel("PDF")
plt.title("Predicted PDFs")
plt.grid(True, alpha=0.3)
plt.show()

Context window: 60
DOF: 16
Device: cuda
Train X: torch.Size([402426, 60]), dtype: torch.float32
Train Y: torch.Size([402426, 1]), dtype: torch.float32
LSTM initialized: [128, 64, 32], dropout=0.02, L2=0.002


C:\Users\MainUser\miniconda3\envs\cs231_env\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.02 and num_layers=1
  warnings.warn(


RuntimeError: expected scalar type Double but found Float